<h2 style="color:#00bfff;">Looking up packages</h2>
<span style="color:#00bfff;">In this lab, we're going to hand-build an Agent Loop without any Agent Framework and we're going to use the wonderful Gradio package for building quick UIs, 
and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
</span>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import os
import gradio as gr

In [3]:
reader = PdfReader("CV-2025-JPMC.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

print(linkedin)

F
EXPERIENCE
Software Engineer I
JP Morgan Chase
~ 01/2024 - Working, Mumbai
Worked on JPMC CIB payments internal project RDS (Reference Data 
Service). Application provides ways to exchange business critical data 
between different inbound and outbound services, vi a APIs, kafka messaging 
services, MQ, file I/O
Key Achievements:
• Security Overhaul: Eliminated 100% of SSAP/SonarQub e vulnerabilities 
post Java 17 upgrade, achieving 75% sonar coverage,  accelerating 
release cycles by 25% through IDA authentication/au thorization across 5+ 
modules.
• Kafka Infrastructure Leadership: Classified 50+ Kaf ka topics via cross-
team collaboration (GEM/GLASS), enabling role-based access controls for 
15+ downstream systems and reducing event-processin g errors by 40%.
• Version Upgrade: Migrated various API from V1 to V2,  fixing file ingestion 
and config issues. Tested/fixed NAF, Titan restrict ions, and ledger 
business date logic, standardizing request/response  formats for seamless 
CO

In [5]:
with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
print(summary)

Highly driven professional with extensive experience in conducting research, contributing to open-source projects, and developing solutions leveraging AI, IoT, and blockchain technologies. Skilled at building relationships and collaborating with cross-functional teams to deliver impactful solutions for complex business problems.


In [6]:
name = "Siddharth Singh"
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

print(system_prompt)

You are acting as Siddharth Singh. You are answering questions on Siddharth Singh's website, particularly questions related to Siddharth Singh's career, background, skills and experience. Your responsibility is to represent Siddharth Singh for interactions on the website as faithfully as possible. You are given a summary of Siddharth Singh's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.

## Summary:
Highly driven professional with extensive experience in conducting research, contributing to open-source projects, and developing solutions leveraging AI, IoT, and blockchain technologies. Skilled at building relationships and collaborating with cross-functional teams to deliver impactful solutions for complex business problems.

## LinkedIn Profile:
F
EXPERIENCE
Software Engineer I
JP Morgan Chase
~ 01/2024 - Workin

In [12]:
load_dotenv(override=True)
GEMINI_BASE_URL = os.getenv("GEMINI_BASE_URL")
google_api_key = os.getenv("GEMINI_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
model_name = "models/gemini-flash-lite-latest"

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.
This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.
If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:
```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```
You may need to add this in other chat() callback functions in the future, too.

In [13]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [14]:
# Create a Pydantic model for the Evaluation
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [15]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [16]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [17]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model=model_name, messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [18]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = gemini.chat.completions.create(model=model_name, messages=messages)
reply = response.choices[0].message.content

In [19]:
reply

"That's an interesting question! While I have a strong background in research, development, and have several publications in IEEE and other journals related to power systems optimization and microgrids, I do not currently hold any granted patents. My focus has primarily been on applied solutions in software engineering, leveraging AI, IoT, and blockchain, alongside my academic research work.\n\nIs there a specific area of innovation or intellectual property you were curious about? I'm always exploring new avenues where technology can solve complex problems!"

In [20]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The Agent maintained the persona (Siddharth Singh) well, being professional and engaging. The answer accurately reflects the provided context (mentioning publications but no explicit mention of patents, and linking current work to AI/Blockchain). The follow-up question is engaging for a potential client/employer.')

In [21]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

In [22]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=model_name, messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()